In [5]:
import pandas as pd
import json
from pathlib import Path
import seaborn as sns
import plotly.express as px

In [3]:
archivo = Path("datos/historico_mesas_operativas_2026-09-08_1002.xlsx")
salida = Path("datos/especies.json")

df = pd.read_excel(archivo, sheet_name="Detalle por especie")

resultado = (
    df.groupby("Especie", dropna=False)["Cajas a inspección"]
      .sum()
      .reset_index()
      .rename(columns={"Especie": "especie", "Cajas a inspección": "cajas"})
)

resultado["especie"] = resultado["especie"].fillna("DESCONOCIDO").astype(str).str.strip()
resultado["cajas"] = resultado["cajas"].fillna(0).round().astype(int)
resultado = resultado.sort_values("cajas", ascending=False)

datos = resultado.to_dict(orient="records")

with open(salida, "w", encoding="utf-8") as f:
    json.dump(datos, f, ensure_ascii=False, indent=4)

print(f"Generado: {salida}")
print(f"Especies: {len(datos)}")

Generado: datos\especies.json
Especies: 47


In [6]:
df_filtrado = df[
    (df["Año"] == 2025) &
    (df["Sitio"] == "VALPARAISO")
]

# Seleccionar 10 especies aleatoriamente
especies = (
    df_filtrado["Especie"]
    .dropna()
    .unique()
)

especies_aleatorias = pd.Series(especies).sample(
    n=10,
    random_state=42
).tolist()

df_filtrado = df_filtrado[
    df_filtrado["Especie"].isin(especies_aleatorias)
]

# Agrupar producción mensual
datos = (
    df_filtrado
    .groupby(["Especie", "Mes"])["Cajas a inspección"]
    .sum()
    .reset_index()
)

# Convertir los meses en columnas
tabla = datos.pivot(
    index="Especie",
    columns="Mes",
    values="Cajas a inspección"
).fillna(0)

# Mantener solamente los meses disponibles
meses = ["Noviembre", "Diciembre"]

tabla = tabla.reindex(columns=meses, fill_value=0)

# Convertir Especie nuevamente en columna
tabla = tabla.reset_index()

print(tabla)

Mes    Especie  Noviembre  Diciembre
0          AJO        0.0     8400.0
1     ARANDANO        0.0    34128.0
2       CEREZA     7000.0     9742.0
3    CHIRIMOYA      500.0        0.0
4      CIRUELA        0.0    31952.0
5      DURAZNO    40051.0    24176.0
6    MANDARINA     4000.0        0.0
7      NISPERO     1000.0        0.0
8        PALTA   117065.0    79905.0
9       PEONIA     8867.0        0.0


In [7]:
fig = px.parallel_coordinates(
    tabla,
    dimensions=[
        "Noviembre",
        "Diciembre"
    ],
    color="Diciembre",
    labels={
        "Noviembre": "Noviembre",
        "Diciembre": "Diciembre"
    },
    title="Producción por especie - Valparaíso 2025"
)

fig.show()